# IT'S ALIVE: A Virtual Worm Driven by Its Real Brain

This notebook demonstrates the complete pipeline: **real C. elegans connectome data → spiking neural network → physical body → emergent behavior**.

We poke the virtual worm and watch the touch stimulus cascade through 299 real neurons to drive muscle contractions and body movement — all from the worm's actual wiring diagram.

In [ ]:
import sys
sys.path.insert(0, "../creatures-core")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from matplotlib.collections import LineCollection
import matplotlib.gridspec as gridspec

from creatures.connectome.openworm import load
from creatures.connectome.types import NeuronType
from creatures.neural.brian2_engine import Brian2Engine
from creatures.neural.base import NeuralConfig
from creatures.body.worm_body import WormBody
from creatures.body.base import BodyConfig
from creatures.experiment.runner import SimulationRunner, CouplingConfig

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

In [ ]:
# Build the complete system: connectome → neural network → body → coupling
connectome = load("edge_list")
engine = Brian2Engine()
engine.build(connectome, NeuralConfig(weight_scale=3.0, tau_syn=8.0, tau_m=15.0))
body = WormBody(BodyConfig(dt=1.0))
body.reset()

runner = SimulationRunner(engine, body, CouplingConfig(
    firing_rate_to_torque_gain=0.004,
    inhibitory_gain=-0.002,
    poke_current=50.0,
    poke_duration_ms=50.0,
))
print(f"System ready: {connectome.n_neurons} neurons → {connectome.n_synapses} synapses → 12-segment body")

## Run the Simulation

Poke the worm's posterior body (segment 8) and watch the neural cascade drive movement.

In [ ]:
# Settle, then poke
runner.run(20)  # let body settle
runner.poke("seg_8", (0, 0.15, 0))  # poke posterior
runner.run(300)  # observe response

# Extract traces
times = [f.t_ms for f in runner.frames]
coms_x = [f.body_state.center_of_mass[0] for f in runner.frames]
coms_y = [f.body_state.center_of_mass[1] for f in runner.frames]
n_active = [len(f.active_neurons) for f in runner.frames]
n_muscles = [len(f.muscle_activations) for f in runner.frames]

peak_frame = max(runner.frames, key=lambda f: len(f.active_neurons))
displacement = ((coms_x[-1] - coms_x[0])**2 + (coms_y[-1] - coms_y[0])**2)**0.5

print(f"Simulation: {times[-1]:.0f}ms")
print(f"Peak neural activity: {len(peak_frame.active_neurons)} neurons at t={peak_frame.t_ms:.0f}ms")
print(f"Total displacement: {displacement:.4f} units")
print(f"Muscles engaged: {max(n_muscles)}")

## The Full Picture: Brain Activity + Body Movement

Four synchronized panels showing the complete sensorimotor loop.

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(3, 2, height_ratios=[1.5, 1, 1], hspace=0.35, wspace=0.3)

# Panel 1: Worm trajectory (top-left)
ax1 = fig.add_subplot(gs[0, 0])
# Color trajectory by time
points = np.array([coms_x, coms_y]).T.reshape(-1, 1, 2)
segments = np.concatenate([points[:-1], points[1:]], axis=1)
norm = plt.Normalize(times[0], times[-1])
lc = LineCollection(segments, cmap="viridis", norm=norm, linewidth=3)
lc.set_array(np.array(times[:-1]))
ax1.add_collection(lc)

# Mark start and poke time
ax1.plot(coms_x[0], coms_y[0], "o", color="green", markersize=10, label="Start", zorder=5)
poke_idx = min(range(len(times)), key=lambda i: abs(times[i] - 20))
ax1.plot(coms_x[poke_idx], coms_y[poke_idx], "x", color="red", markersize=12,
         markeredgewidth=3, label="Poke", zorder=5)
ax1.plot(coms_x[-1], coms_y[-1], "s", color="blue", markersize=10, label="End", zorder=5)

ax1.set_xlabel("X position")
ax1.set_ylabel("Y position")
ax1.set_title("Worm Trajectory (color = time)", fontweight="bold")
ax1.legend(fontsize=9)
ax1.set_aspect("equal")
margin = max(max(coms_x) - min(coms_x), max(coms_y) - min(coms_y), 0.01) * 0.3
ax1.set_xlim(min(coms_x) - margin, max(coms_x) + margin)
ax1.set_ylim(min(coms_y) - margin, max(coms_y) + margin)
plt.colorbar(lc, ax=ax1, label="Time (ms)")

# Panel 2: Body snapshots (top-right)
ax2 = fig.add_subplot(gs[0, 1])
snapshot_times = [20, 50, 100, 150, 200, 300]
colors_snap = plt.cm.plasma(np.linspace(0, 1, len(snapshot_times)))
for t_target, color in zip(snapshot_times, colors_snap):
    idx = min(range(len(times)), key=lambda i: abs(times[i] - t_target))
    frame = runner.frames[idx]
    xs = [p[0] for p in frame.body_state.positions]
    ys = [p[1] for p in frame.body_state.positions]
    ax2.plot(xs, ys, "-o", color=color, markersize=3, linewidth=2, alpha=0.8,
             label=f"t={times[idx]:.0f}ms")

ax2.set_xlabel("X")
ax2.set_ylabel("Y")
ax2.set_title("Body Shape Over Time", fontweight="bold")
ax2.legend(fontsize=8, ncol=2)
ax2.set_aspect("equal")

# Panel 3: Neural activity (middle-left)
ax3 = fig.add_subplot(gs[1, 0])
ax3.fill_between(times, n_active, alpha=0.3, color="#2196F3")
ax3.plot(times, n_active, color="#2196F3", linewidth=1)
ax3.axvline(20, color="red", linestyle="--", alpha=0.5, label="Poke")
ax3.set_xlabel("Time (ms)")
ax3.set_ylabel("Active neurons")
ax3.set_title("Neural Activity", fontweight="bold")
ax3.legend()

# Panel 4: Muscle engagement (middle-right)
ax4 = fig.add_subplot(gs[1, 1])
ax4.fill_between(times, n_muscles, alpha=0.3, color="#FF5722")
ax4.plot(times, n_muscles, color="#FF5722", linewidth=1)
ax4.axvline(20, color="red", linestyle="--", alpha=0.5, label="Poke")
ax4.set_xlabel("Time (ms)")
ax4.set_ylabel("Active muscles")
ax4.set_title("Muscle Engagement", fontweight="bold")
ax4.legend()

# Panel 5: Displacement over time (bottom-left)
ax5 = fig.add_subplot(gs[2, 0])
displacements = [((x - coms_x[0])**2 + (y - coms_y[0])**2)**0.5 for x, y in zip(coms_x, coms_y)]
ax5.plot(times, displacements, color="#4CAF50", linewidth=2)
ax5.axvline(20, color="red", linestyle="--", alpha=0.5, label="Poke")
ax5.set_xlabel("Time (ms)")
ax5.set_ylabel("Displacement (units)")
ax5.set_title("Body Displacement from Start", fontweight="bold")
ax5.legend()

# Panel 6: Joint angles heatmap (bottom-right)
ax6 = fig.add_subplot(gs[2, 1])
joint_data = np.array([f.body_state.joint_angles for f in runner.frames]).T
im = ax6.imshow(joint_data[:, ::5], aspect="auto", cmap="RdBu_r",
                extent=[times[0], times[-1], 10.5, -0.5],
                vmin=-0.3, vmax=0.3)
ax6.axvline(20, color="red", linestyle="--", alpha=0.5)
ax6.set_xlabel("Time (ms)")
ax6.set_ylabel("Joint index")
ax6.set_title("Joint Angles (blue=dorsal, red=ventral)", fontweight="bold")
plt.colorbar(im, ax=ax6, label="Angle (rad)")

fig.suptitle("Virtual C. elegans: Real Connectome → Spiking Network → Physical Body → Movement",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()